# Introduction

Landscapes are a reflection of the processes and properties that shape
them, but inverting topographic data for quantitative geomorphic
information remains challenging (e.g
\[cite:@barnhartInvertingTopographyLandscape2020\]\[cite:@robertsUpliftHistoryColorado2012\]
). There are a variety of challenges in this process, but one of them is
that geomorphic signals manifest themselves in topography in subtle and
complex ways that simple metrics struggle to quantify; metrics that can
capture spatial structure of topography are needed. At the same time,
the advent of new remote sensing technologies, particularly lidar, has
led to the creation of high resolution topographic data with large
spatial extents, while data on subsurface properties that shape
landscapes remain scarce and discontinuous. Deep learning methods, large
neural networks supplied with large amounts of data, have been
successful in capturing spatial patterns from microscopic
\[cite:@iglesiasDeepLearningDiscrimination2019\] to global scales
\[cite:@priceGenCastDiffusionbasedEnsemble2024\]. Deep learning methods
like convolutional neural networks (CNNs) have been used in
geomorphology in tasks like feature
detection\[cite:@fairfaxEEAGERNeuralNetwork2023\]\[cite:@maxwellExploringInfluenceInput2023\],
hazard
mapping\[cite:@liArtificialIntelligenceincorporatedPrediction2025\], and
the processing \[cite:@RSCNNCNNBasedMethod\] and
classification\[cite:@maxwellExploringInfluenceInput2023\] of remotely
sensed data, but can they learn the complex spatial signals in
topography that relate to hard to quantify subsurface properties?
Training a network to complete that task successfully outside of its
training set would mean a network that has uncovered a theoretical
geomorphic relationship, which remains a frontier in geomorphology.
There are a number of reasons why this remains challenging, but one
aspect is the belief, born out in application, that neural networks are
"black boxes" and are unable to help us as scientists gain new insights
into our problems. Recent work with methods like "explainable AI"
\[cite:@dahalExplainableArtificialIntelligence2023\]
\[cite:@youssefLandslideSusceptibilityModeling2023\]are starting to fill
that gap, but it is still a significant challenge to take a trained
neural network and use it to help inform our theory of the physical
world. In this paper we demonstrate how a convolutional neural network
trained to invert a classic physics based numerical model of
geomorphology can be interrogated to understand what about geomorphology
a neural network has "learned". We ran the streampower-diffusion
landscape evolution model across parameter space and trained a neural
network to invert the equilibrium topography of the model into the
parameters of that model run. We then not only evaluate the networks
performance, but we identify which spatial patterns are successfully
encoded in the neural network, and which patterns the network has failed
to learn.

## Landscapes as advective/diffusive systems

To test the extent to which a neural network can extract geomorphically
meaningful information from topography and to study the interpretability
of the network's learning, we use simulated topography generated by a
process based numerical model. Real topographic data presents a host of
challenges, such as differences in collection conditions, differences in
processes and properties impacting landscapes, and a lack of well
constrained parameters that control landscape evolution to train a
network to extract. With numerical models, the generation of the
topography is controlled and the "answer", that is the parameters used
to generate the topography, are known exactly, allowing us to evaluate
network performance and networks geomorphic learning. This serves as a
"first test" on training neural networks to solve geomorphic problems
before moving to real data. In this work we use a simple 3-parameter
numerical model, based on the "stream power" formulation for channel
erosion together with a diffusion equation to represent downhill soil
transport, to generate a dataset of synthetic landscapes with varying
relative strengths of water versus gravitational erosion. This
theoretical model for landscape evolution, which we refer to as the
Stream Power plus Diffusion (SPD) model, is simple in formulation while
still creating landscapes that are complex and realistic. While the SPD
model omits many of the factors that shape real landscapes, it captures
the competition between fluvial and hillslope transport that is thought
to be a key element in the dynamics of river-carved landscapes. This
treatment of landscape evolution as a simple advection-diffusion system
is common in the geomorphology literature and has been extensively
studied \[cite:@howardModelingFluvialErosion1994\]
\[cite:@whippleDynamicsStreampowerRiver1999\]\[cite:@tuckerTopographicOutcomesPredicted2002\]\[cite:@tuckerModellingLandscapeEvolution2010\]\[cite:@perronControlsSpacingFirstorder2008\]\[cite:@theodoratosScalingSimilarityStreampower2018\]
The SPD model describes changes in elevation $z(x,y,t)$ across time and
is formulated as
$$\frac{\partial z}{\partial t}= D\nabla^2 z - KA^m|\nabla z|^n +U$$
where $D$ is a diffusivity-like gravitational soil transport
coefficient, $K$ is the fluvial erosion coefficient, $U$ is the uplift
rate relative to a given base level, $A$ is the upstream contributing
drainage area, $\nabla z$ is the gradient magnitude in the steepest
downhill direction, and $m$ and $n$ are constants that depend on the
specific derivation of the streampower equation. In this paper we use
values 0.3 and 0.7 for $m$ and $n$ respectively, which derives from the
assumption that the rate of fluvial erosion is proportional to bed shear
stress \[cite:@howardModelingFluvialErosion1994\]. The SPD mode is
effectively a nonlinear advection-diffusion equation with a source term.

The equation can be solved numerically to get an elevation field at a
given point in time. $D$ and $K$ are lumped parameters that describe how
the landscapes respond to hillslope and river processes, including
climatic factors and the underlying geology. Inferring the ratio of
these parameters is an exercise akin to inferring information about
material properties from topography. While we can estimate these
parameters with methods like ridgeline curvature
\[cite:@roeringEvidenceNonlinearDiffusive1999\], slope-area plots
\[cite:@wobusTectonicsTopographyProcedures2006\], or chi plots,
\[cite:@perronIntegralApproachBedrock2013\], inferring these parameters
with a neural network is a first step to estimating material parameters
where an existing topographic relationship does not exist. The ratio of
$K$ to $D$ (which can be cast as a type of Péclet number) represents the
relative strength of channelized fluvial incision (an advection process)
to gravitational soil transport (a diffusive process), and it is a key
control on the landscape that develops. Perron et al. 2008, 2009, 2012
showed that the $\frac{K}{D}$ ratio determines the transition from
smooth hillslopes to branching valley networks, and sets the scale of
first-order drainages and the hillslope-valley transition
\[cite:@perronControlsSpacingFirstorder2008\]\[cite:@perronFormationEvenlySpaced2009\]\[cite:@perronRootBranchingRiver2012\].
Similarly, Theodoratos et al. 2018 use a Péclet-like ratio to
investigate characteristic scales in landscapes and understand how they
relate to the fate of perturbations in the landscapes
\[cite:@theodoratosScalingSimilarityStreampower2018\]. A common theme
across these and other scaling analyses is that the ratio of
gravitational transport efficiency, $D$, to fluvial erosion efficiency,
$K$ exerts a fundamental control on landscape texture. In this work we
test the ability of a trained neural network to infer the ratio $K/D$
from synthetic digital topography, for which that ratio is independently
known.

## Neural Networks

Neural networks are a machine learning method that allows many
relatively simple functions to approximate a more complex one.
Theoretically a large enough neural network can approximate any function
with arbitrary precision
\[cite:@cybenkoApproximationSuperpositionsSigmoidal1989\]; in practice
the main limitations often lie in constructing a network that is large
enough, and collecting enough data to train it. While neural networks as
a methodology date to the 1940's, they have found significant success
recently due to technological innovations allowing the rapid training of
large networks on large amounts of data, as well as the creation of
massive datasets to train the network with. A "traditional" feed-forward
neural network involves a large number of linear functions ("neurons",
or "layers") chained together with "activation functions", simple
functions like a logistic function that introduce nonlinearity to the
network. The parameters of each neuron, or "weights", are initialized
randomly, and tuned during the training process, where the output of the
network for a given input is compared to the true value. The mismatch
between these values, or "loss" is then used to guide the modification
of the weights via gradient descent
\[cite:@princeUnderstandingDeepLearning2023\]. Convolutional neural
networks (CNN's) have proven to excel at problems related to image
processing and spatial data. In a CNN, some or all of the layers have
been replaced by convolutions: windows of weights that sweep over the
(typically but not necessarily) 2D input and returning the weighted sum
of the input that overlaps with the window
[fig:apdx:conv](fig:apdx:conv). The output of a convolution is the same
dimension as the input so, along with activation functions, the output
is typically flattened and fed into a traditional neural network to
produce a single number, in the case of regression networks, or a list
of class probabilities, for classification problems.

In this paper we create a simple regression neural network with
convolutions and train it to infer the ratio of D and K for synthetic
landscapes generated using an SPD numerical model. Then we interpret
what patterns the network has and has not learned.

## Numerical models

To generate training data for the network we solve the
streampower-diffusion equation in two dimensions for a realistic range
of diffusivity and incision coefficients. Model runs are given one of
ten initial random terrains to generate a dataset of 9,000 landscapes
(Figure [fig:performance](fig:performance) b-c). The model domain is 500
m by 1500 m with 5m grid cells following
[cite:&perronControlsSpacingFirstorder2008](cite:&perronControlsSpacingFirstorder2008).
This yields landscapes with a central ridge and parallel river valleys.
By ensuring that our landscapes have a consistent structure, we can
focus our interpretation work on looking for these patterns in the
trained network (see table [tab:parameters](tab:parameters) for the full
parameter details). By choosing a realistic range of parameters for $D$
and $K$ separately, we get distinct non-uniform distributions for $D/K$
and $\frac{K}{D}$ [fig:apdx:dist](fig:apdx:dist) which when used as
targets for the neural network result in differences in performance
because of how to distributions interact with the mean squared error
function used to guide training
\[cite:@hodsonRootmeansquareErrorRMSE2022\].

## Convolutional Neural Network

Convolutional Neural Networks are neural networks with relatively simple
components commonly used for image recognition tasks
\[cite:@qinHowConvolutionalNeural2018\]. In this paper we use a CNN
designed to be as small as possible while still being able to extract
some relatively complex spatial features. The network has the following
architecture: three convolutional units, a flattening layer, followed by
three linear units that reduce the output to a single number
[fig:apdx:nn-arch](fig:apdx:nn-arch). Each convolutional unit contains a
2D convolution layer, a max pooling layer, that resamples the output to
half resolution with a maximum-value resampling, and a ReLU activation
layer, that sets all negative output values to zero, introducing
nonlinearities into the network. Each convolution has a different window
size, and expects and produces an output with a different number of
bands. The first layer has a three-by-three convolution (or window of
weights), takes a 1-band input (either the landscape elevation field or
a quantity derived from it) and creates a 10-band output. The second
layer has a five-by-five convolution, takes a 10-band output (from the
previous convolutional unit) and has a 20-band output. The last layer
has a seven-by-seven convolution, takes a 20-band input and has a 20
band output. This 20-band output is then flattened into a 1D vector,
which is passed through a linear transformation with 100 outputs and
subjected to a ReLU activation layer. This is then passed to a linear
transformation and ReLU with 10 outputs , and finally a linear
transformation that produces a single number output, the is interpreted
as the network's inferred $D/K$ value for the given input image.

### Training

This neural network is a function that takes in a 2D single band input,
and produces a single number (the target). Our experiments have 4
possible inputs, model elevation, slope, curvature, and flow
accumulation, and four possible targets, $D/K$, $K/D$, $\log_{10}(D/K)$,
and $\log_{10}(K/D)$ for a total of 16 different input/output pairs.
Additionally for each pair we train 4 networks, each with a different
set up of randomized initial weights, for 64 total neural networks.
Training multiple networks with different initial weights helps us
ensure that results aren't biased by a specific starting point. The
selection of inputs allow us to understand what topographic information
is most useful, and the selection of different targets allow us to
explore how difference distributions of the target variable change
performance. Each network is trained on a subset of the collection of
model runs with 7200 individual landscapes, and is tested on the
remaining 1800 landscapes not used for training. All target variables
and input datasets are normalized by the standard deviation of the
training subset.

# Results

The network trained to infer $D/K$ from the elevation data performs
well, with a range normalized mean standard error of 0.02075 (Figure
[fig:performance](fig:performance)).

[file:figs/performance_raw.pdf](figs/performance_raw.pdf)

Performance is best for high $D/K$ landscapes with increased spread and
overestimation where $D/K$ is low. We interpret this reduced performance
at low $D/K$ as reflecting the spatial resolution in our models. The
$D/K$ ratio effectively sets the hillslop length
\[cite:@howardModelingFluvialErosion1994\]
\[cite:@tuckerHillslopeProcessesDrainage1998\]. At sufficiently low
$D/K$ the hillslope length becomes shorter than the resolution of the
grid, and the simulated landscapes no longer appear as distinct as $D/K$
is reduced further. When looking at performance across other data types
(Figure [fig:comparison](fig:comparison)) we see significant increases
in performance when the network is given slope or curvature data, with
the best performance (NRMSE 0.00408) coming from curvature data.

[file:figs/perf_compare.pdf](figs/perf_compare.pdf)

Flow accumulation data decreases performance when inferring $D/K$,
although this is likely due to the extreme skew to the distribution of
this data, as performance increases when the network is given the log of
the flow accumulation data. However even raw flow accumulation data
*increases* performance of the network trained to infer $\frac{K}{D}$
The inputs to the neural network are equilibrium landscapes, were the
time derivative of elevation is zero, and so are solutions to
$(D/K)\nabla^2 z+A^m |dz/ds|^n=U/K$ implying that $D/K$ is a function of
slope and curvature and more closely related to these topographic
derivatives than elevation
\[cite:@theodoratosScalingSimilarityStreampower2018\], which could be
driving the discrepancy in performance.

# What has the network learned?

The performance of the network leads us to believe that the network may
have learned landscape patterns that are geomorphically meaningful.
Remember that the network takes in a landscape, and transforms it into
something it can correlate to $D/K$, which is a key control of the
landscape morphology. But what does it transform the landscape into? Are
these patterns that are meaningful to us as geomorphologists? Can we
interpret the network in a theoretically grounded way? To answer these
questions, we need to interrogate the network, which we can do in a
variety of ways.

## Activation Maximization

A tool commonly used for the interpretation of convolutional neural
networks is "activation maximization", where you construct an input that
will "maximize" the output of the network, or part of the network
\[cite:@qinHowConvolutionalNeural2018\]. When we construct activation
maximization images for specific neurons or convolutions in the network,
the image can be thought of as the pattern that a specific neuron is
most sensitive to. These are constructed through gradient ascent much in
the same way a network is trained; an initial input of random noise is
created, and then is tweaked, as guided by the gradients of the network,
towards producing the highest output (hence *ascent* instead of descent,
which is used in training). For this network, the activation
maximization images do not resemble landforms, but instead appear more
similar to a television tuned to the wrong channel (Figure
[fig:apdx:actmax](fig:apdx:actmax)). However some structure seems
apparent. Many of the images have a systematic wavelength pattern, and
there appears to be a consistent wavelength and orientation. One
qualitative interpretation is that this could be akin to a tightly
spaced system of valleys, as our model domain was designed to emphasize
the connection between $D/K$ and valley spacing
\[cite:@perronControlsSpacingFirstorder2008\].

## Valley Spacing

To analyze how the trained network responds to repeated shapes with a
fixed frequency, such as quasi-evenly spaced valleys, we created a
dataset of "pseudo landscapes" generated from sine waves. This allows us
to test the network on "landscapes" that have clear (and known) valley
spacing, but without other features. We take our network that was
*trained* on elevation data of landscapes generated by the
streampower-diffusion model and give it these "sine landscapes" to see
what $\log_{10}_{}(D/K)$ value the network would assign (Figure
[fig:sine](fig:sine)). Because we are evaluating the network's response
to out-of-sample data with a completely different generating process we
use the model that was trained on a log-normalized target which is more
robust to out-of-sample data (section
[#sect:apdx:targets](#sect:apdx:targets)) It should be noted that these
sine landscapes have no true $\log_{10}(D/K)$ as they are not generated
by landscape processes, however we can test the networks "geomorphic
intuition" about the relationship between valley spacing and
$\log_{10}(D/K)$. The results show an increasing correlation between
valley spacing and inferred $log_{10}(D/K)$ after valley spacing reaches
about 20 meters (4 grid cells), with a negative correlation before 20
meters, and a slight break at 50 meters. The correlation suggests that
the model is indeed sensitive to wavelength of terrain features, with
longer wavelengths leading to larger inferred values of $D/K$.

[file:figs/valley_perf.pdf](figs/valley_perf.pdf)

## Drainage Networks

Natural hillslope-valley landscapes vary widely in their drainage
density, which is conventionally defined as the average total length of
channels per unit area. Although our SPD model does not discriminate
explicitly between channeled and con-channeled locations, the density of
valley-like features simulated by the model does vary systematically
with the D/K ratio, reflecting the competition between diffusion and
channel incision \[cite:@howardBadlandMorphologyEvolution1997\]
\[cite:@tuckerHillslopeProcessesDrainage1998\] . To test the trained
CNN's sensitivity to the density of valley (channel-formed) features, we
construct a series of inputs that contain only information about the
drainage network, omitting any elevation data. These network maps are
generated by taking a highly dissected modeled landscape elevation field
and thresholding its flow accumulation data at a range of different
thresholds, creating a series of binary masks in which 0 represents a
pixel above the accumulation threshold (a "channel") and 1 represents a
pixel below the accumulation threshold (a "hillslope"). A low-threshold
image will include many pixels in the channel category, and as a result
it will appear as a dense drainage network, whereas a high-threshold
image will contain only the highest-order streams in the landscape. When
we take the network that was trained on the *elevation* data of the
streampower-diffusion model landscapes and ask it to assign $D/K$ values
from these binary masks of river systems, we see an inverse relationship
between inferred $D/K$ and the input drainage density (Figure
[fig:drain-perf](fig:drain-perf)). This finding indicates that the CNN
trained on elevation data is sensitive to drainage network density even
when that density is represented by a simple binary mask. Like with the
valley spacing experiment, there is a break in the relationship around
where $log_{10}(D/K)$ is about -3. In both cases the cause of the break
remains unexplained, but could be due to nonlinearities in the learned
relationsip or some sort of perceived threshold in the valley spacing or
drainage density inputs. Further work is needed to better understand the
cause of the break in relationship.

[file:figs/drainage_density.pdf](figs/drainage_density.pdf)

# What has the network failed to learn?

While it appears the network has "learned" some geomorphology, it is
inherently limited by the data used to train it. Not only does this mean
it is limited to a pure advective/diffusive world, but it has seen a
limited slice of that world, most immediately clear in the spatial
boundary of the models. All of the landscapes the network has seen are
long rectangles with a central ridge and mostly parallel valleys
perpendicular to the ridge. These boundary position impose an overall
structure to the modeled topography. While that structure allows makes
network interpretation easier, by allowing us to look for things like
valley spacing, we want to test how model performance changes when that
structure is altered, to understand if it is overreliant on it. We
tested this by taking our modeled landscapes and "swapping them" either
with a split down the center to mimic a central valley or by tiling the
landscape and randomly shuffling the tiles (Figure
[fig:apdx:shuffex](fig:apdx:shuffex)). When the network trained to infer
$\log_{10}D/K$ from elevation is given the shuffled and swapped
landscapes it is no longer able to infer the target value. In both cases
it systematically predicts $\log_{10}D/K$ values close to zero from low
$D/K$ landscapes and underpredicts high $D/K$ landscapes by up to two
orders of magnitude (Figure [fig:shuf-perf](fig:shuf-perf)). This
mismatch in performance could be because low $D/K$ landscapes have short
hillslope lengths which are preserved within tiles, while high $D/K$
landscapes have their hillslope length broken during shuffling.
Additionally shuffling will introduce discontinuities which in real
landscapes with high relative diffusivities would be "smoothed" out by
diffusive patterns. There is a systematic striping in the "swapped"
landscape set which corresponds to different values of K, that is, one
stripe represents landscapes with the same K but different D. This means
that while the network is not able to reconstruct the exact ratio of
$D/K$ after the model output is swapped, it does retain some ability to
recognize different landscapes. When the network trained to infer
$log_{10}(D/K)$ from slope is given the shuffled and swapped data, it
still performs very strongly. The increased performance of this model on
both the test set and the shuffled and swap inputs implies it has
learned more robust patterns than the model trained on elevation.

[file:figs/shuff_perf_paper.pdf](figs/shuff_perf_paper.pdf)

# Inputs and Targets

The choice of what input to use (elevation, slope, curvature, or flow
accumulation) as well as which output target to supply the neural
network $D/K$ or $K/D$ has significant impacts on the final performance,
although all options demonstrate some learning by the neural network.
All else being equal, the neural network performs better when trained
with using topographic derivatives that are geomorphically
meaningful–and relate to the governing equations. Not only does
performance increase on the test set, but the network is not "confused"
when the input is shuffled around, as it is when given elevation. This
could be because slope and curvature are more evenly distributed across
the landscape, which is not the case for elevation. Prior work has found
performance increases when neural networks trained on topographic tasks
are given topographic derivatives
\[cite:@maxwellExploringInfluenceInput2023\], but more work remains to
be done to understand when a specific derivative is, or is not, helpful.
Do slope and curvature increase performance because they relate to the
governing equation involved in the task, or because of the differences
in their spatial distributions? When moving from elevation to slope, the
data is in some sense more "geomorphically meaningful", but information
about the absolute value of elevation is lost. Will slope always perform
better than elevation or are there tasks where this information is
critical, like in landscapes where there is an elevation dependent
geomorphic process?

As noted above, a 3x3 convolution should be able to approximate slope or
curvature, and so with longer training time it is possible that the
performance of the network using the elevation dataset would approach
that of the networks trained on slope or curvature. It is possible that
a different architecture, like an initial single 3x3 convolution to
learn slope prior to the first 10 channel convolutional layer would be
necessary as well. Additionally when using a derivative with a very
skewed distribution of values, like flow accumulation, normalization
should be considered to help the network train more efficiently. A
discrepancy in performance between different ratio targets is to be
expected, as the distribution of the target variables will be different,
and certain distributions will be harder or easier for a neural network
to learn. This could be mitigated with normalization, although while in
our experiments log normalization equalized performance between $D/K$
and $K/D$, both underperformed un-normalized $D/K$. While it is
unsurprising that $\frac{K}{D}$ leads to worse performance than $D/K$,
it is not intuitive that would display a stronger preference for flow
accumulation data. This could be due to similarities in distribution
between the input and the target, but should be further investigated.

# Conclusions

This study demonstrates how neural networks can be used not simply to
automate mapping tasks, but as hypothesis generators and testers. Deep
learning methods not only excel at geomorphic tasks, but can be tested
in ways that illuminate what spatial patterns the network uses for its
performance, and in this case study, reveal an "understanding" that
matches the intuition of a geomorphologist. This study shows that these
methods, both neural networks and interpretability techniques are
powerful, and should be applied to domains where our understanding of
signals in topography and their relationship to property and process is
unclear. A natural next step is to look at a similar problem, but with
real data. This could be done with data alone or using a network trained
on numerical models like this one. Such work would have to overcome a
number of challenges, like the collection of a large number of well
constrained and comparable geomorphic parameters (like $K$ and $D$
values), more realistic and variable model domains (i.e. ones that do
not have a strong central ridge), and an accounting of the noise that
comes along with real data. This paper also demonstrates that neural
networks cannot be used without an understanding of physical theory, and
without interpretability work; the trained neural network is *not* an
algorithm that infers process parameter values from the landscape, it is
a collection of geospatial relationships that can be poked, prodded, and
queried to find where it rhymes, and where it doesn't, with our physical
understanding of earth systems.

# References

# Appendix

## The Neural Network

[file:figs/conv.pdf](figs/conv.pdf)

[file:figs/network_viz.pdf](figs/network_viz.pdf)

## Parameters

| Rows (cells) | Columns (cells) | Grid Spacing (m) | Runtime (years) | Timestep (years) | Uplift Rate ($\frac{m}{year}$) | Diffusivity ($\frac{m^2}{year}$) | Streampower K ($\frac{m^{0.4}}{year}$) | Streampower m | Streampower n |
|----|----|----|----|----|----|----|----|----|----|
| 300 | 100 | 5 | 3,000,000 | 500 | 0.001 | 0.005-0.02 | 0.00015-0.002 | 0.3 | 0.07 |

Parameters used for the Streampower-Diffusion model {#tab:parameters}

[file:figs/target_distributions.pdf](figs/target_distributions.pdf)

## Targets

In this work we look at four different targets: $D/K$, $K/D$,
$\log_{10}(D/K)$, and $log_{10}(K/D)$. While for the purposes of this
study the choice of target does not change the fundamental geomorphic
meaning, the distribution of the target variable, and its interaction
with the error function can impact the performance of the machine
learning
method\[cite:@wilhelmHoneyShrunkTarget2020\]\[cite:@nuytsWhenHowTarget2025\].
Because of differences in distributions (Figure
[fig:apdx:dist](fig:apdx:dist)) the differences in performance (Figure
[fig:comparison](fig:comparison)) are not suprising, nor is the fact
that the logarithm effectively equalizes the performance of the two
different ratio targets. However we include and focus on the results of
the un-logged $D/K$ target because of its high performance. While all
our training data is on positive targets, the target variables are
normalized prior to training, and the network is trained to produce a
number from a continuous range that surrounds zero and includes negative
numbers. While for our in-sample data, these negative values produced by
the network become positive, the network has nothing that prevents it
from producing a negative number, although that is meaningless for $D/K$
and $K/D$ values. When we supply the network with inputs that are not
generated from landscape evolution models (sections
[#sec:valley-spacing](#sec:valley-spacing) and
[#sec:drainage-density](#sec:drainage-density)) the networks produce a
coherent pattern of its predictions. but typically with negative
numbers, because the inputs are so distinct from the training data.
While for the purposes of our interpretation, the coherent pattern is
what matters, we present these results as performed on the
$\log_{10}(D/K)$ since for the logged values, negative numbers are
technically meaningful.

## Activation Maximization Output

\[<file:figs/max_6_16.png>\]

## Shuffling

[file:figs/shuffle_viz.pdf](figs/shuffle_viz.pdf)